# Comparativo global: MASTER, TFB, RandomTopJ e benchmarks

Este notebook consolida os JSONs de resultados e, quando houver `sinais.csv` na mesma pasta, incorpora as AUCs direcionais.

**Ponto importante de interpretação:** TFB e MASTER podem ter `pred_len` com significado diferente. Por isso, a comparação principal deve ser feita por `janela_trading`/`horizonte_comparavel` (`k` de rebalanceamento), não apenas por `pred_len`.


In [1]:
from pathlib import Path
import sys
import re
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'utils':
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.comparativo_metricas_global import comparar_global, METRICAS_INTERESSE

OUT = ROOT / 'simulacoes' / 'comparativo_global_master_tfb'
OUT


PosixPath('/sonic_home/igor.viveiros/paralelo/simulacoes/comparativo_global_master_tfb')

In [ ]:
dfs = comparar_global(
    base_dir=ROOT,
    output_dir='simulacoes/comparativo_global_master_tfb',
    top_n=30,
)

metricas = dfs['metricas']
resumo = dfs['resumo']
top_configs = dfs['top_configs']
ranking = dfs['ranking']

neg_master = pd.read_csv(
    "/sonic_home/igor.viveiros/paralelo/simulacoes/master_tfb_experimento/acerto_negativos_master.csv"
)

metricas["output_dir"] = metricas["json_path"].apply(lambda p: str(Path(p).parent))

metricas = metricas.drop(
    columns=["taxa_acerto_negativos", "mean_precision_negative"],
    errors="ignore"
)

metricas = metricas.merge(
    neg_master[[
        "output_dir",
        "taxa_acerto_negativos",
        "mean_precision_negative",
        "n_pred_negativos",
        "n_acertos_negativos",
        "n_janelas_com_negativos",
    ]],
    on="output_dir",
    how="left"
)

def simplificar_dataset_master(nome):
    nome = str(nome)

    # ordem importa: log_return antes de return
    if re.search(r"(__|_|-)log_returns?$", nome):
        return "log_return"
    if re.search(r"(__|_|-)returns?$", nome):
        return "return"
    if re.search(r"(__|_|-)prices?$", nome):
        return "prices"

    return nome

def corrigir_datasets_master(df):
    if df is None or df.empty or "dataset" not in df.columns:
        return df

    df = df.copy()

    if "grupo" in df.columns:
        mask = df["grupo"].eq("MASTER")
        df.loc[mask, "dataset"] = df.loc[mask, "dataset"].apply(simplificar_dataset_master)
    else:
        df["dataset"] = df["dataset"].apply(simplificar_dataset_master)

    return df

metricas = corrigir_datasets_master(metricas)
resumo = corrigir_datasets_master(resumo)
top_configs = corrigir_datasets_master(top_configs)
ranking = corrigir_datasets_master(ranking)
print(f'Linhas carregadas: {len(metricas)}')
print(f'Arquivos salvos em: {OUT}')
metricas.head()


/sonic_home/igor.viveiros/paralelo/utils/auc_direcional.py:87: RuntimeWarning: Mean of empty slice
  "auc_alta_media_janela": float(np.nanmean(auc_alta_janelas)) if auc_alta_janelas else np.nan,
/sonic_home/igor.viveiros/paralelo/utils/auc_direcional.py:88: RuntimeWarning: Mean of empty slice
  "auc_queda_media_janela": float(np.nanmean(auc_queda_janelas)) if auc_queda_janelas else np.nan,
/sonic_home/igor.viveiros/paralelo/utils/auc_direcional.py:87: RuntimeWarning: Mean of empty slice
  "auc_alta_media_janela": float(np.nanmean(auc_alta_janelas)) if auc_alta_janelas else np.nan,
/sonic_home/igor.viveiros/paralelo/utils/auc_direcional.py:88: RuntimeWarning: Mean of empty slice
  "auc_queda_media_janela": float(np.nanmean(auc_queda_janelas)) if auc_queda_janelas else np.nan,
/sonic_home/igor.viveiros/paralelo/utils/auc_direcional.py:87: RuntimeWarning: Mean of empty slice
  "auc_alta_media_janela": float(np.nanmean(auc_alta_janelas)) if auc_alta_janelas else np.nan,
/sonic_home/igor.vi

In [ ]:
# Cobertura por grupo
if not metricas.empty:
    display(metricas.groupby('grupo').size().rename('n_resultados').reset_index())
    display(metricas.groupby(['grupo', 'modelo']).size().rename('n_resultados').reset_index().sort_values(['grupo', 'n_resultados'], ascending=[True, False]).head(50))


In [ ]:
# Tabela principal das métricas de interesse
cols = ['grupo', 'dataset', 'modelo', 'lookback', 'pred_len', 'janela_trading', 'max_assets', *METRICAS_INTERESSE, 'json_path']
tabela = metricas[[c for c in cols if c in metricas.columns]].copy()
tabela.sort_values(['janela_trading', 'mean_spearman_ic'], ascending=[True, False], na_position='last').head(50)


In [ ]:
# Melhores configurações por métrica
top_configs.head(80)


In [ ]:
# Ranking médio simples nas métricas de interesse: quanto menor, melhor.
ranking_cols = ['grupo', 'dataset', 'modelo', 'lookback', 'pred_len', 'janela_trading', 'max_assets', 'rank_medio_metricas_interesse', *METRICAS_INTERESSE, 'json_path']
ranking[[c for c in ranking_cols if c in ranking.columns]].head(50)


In [ ]:
# Resumo agregado por grupo/modelo/janela de trading
resumo.sort_values(['janela_trading', 'mean_spearman_ic_mean'], ascending=[True, False], na_position='last').head(80)


In [ ]:
# Comparação direta por janela de trading e modelo, usando medianas
if not metricas.empty:
    comp = (
        metricas
        .groupby(['janela_trading', 'grupo', 'modelo'], dropna=False)[[c for c in METRICAS_INTERESSE if c in metricas.columns]]
        .median()
        .reset_index()
        .sort_values(['janela_trading', 'mean_spearman_ic'], ascending=[True, False], na_position='last')
    )
    display(comp.head(100))


## Arquivos gerados

- `metricas_global_long.csv`: base longa, uma linha por JSON encontrado.
- `tabela_metricas_interesse.csv`: somente as métricas centrais.
- `resumo_por_modelo.csv`: média/mediana/desvio/contagem por grupo, dataset, modelo e janela de trading.
- `top_configs_por_metrica.csv`: melhores configurações por métrica.
- `ranking_global.csv`: ranking médio simples das métricas de interesse.
